# Creating Bronze Layer Tables

The purpose of this notebook is to create three bronze layer tables for the Retail Sales ETL Pipeline. 

The following steps were completed: 
1. The schema `retail_schema` is created in the `workspace` catalog.
2. Unity Catalog Volume `retail_raw_data` is created to store the raw data.
3. The raw data is loaded into PySpark DataFrames, with the schema enforced.
4. The DataFrames are then stored into their respective bronze Delta tables, following the Simplex pattern.

### Choosing the Workspace and Schema

In [0]:
%sql
USE CATALOG workspace;

CREATE SCHEMA IF NOT EXISTS retail_schema;

USE SCHEMA retail_schema;

In [0]:
%sql
SELECT current_catalog(), current_database()

current_catalog(),current_database()
workspace,retail_schema


### Creating a new Unity Catalog Volume

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS retail_raw_data;

### Creating bronze tables by ingesting raw data

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType, DoubleType
from pyspark.sql.functions import current_timestamp

# Loading Customer raw data
dbutils.widgets.text("customer_data_path", "/Volumes/workspace/retail_schema/retail_raw_data/master_customer.csv")
customer_data_path = dbutils.widgets.get("customer_data_path")

customer_schema = StructType([
  StructField("Customer_ID", StringType(), True),
  StructField("Customer_Name", StringType(), True),
  StructField("Segment", StringType(), True),
  StructField("Country", StringType(), True),
  StructField("City", StringType(), True),
  StructField("State", StringType(), True),
  StructField("Postal_Code", IntegerType(), True),
  StructField("Region", StringType(), True),
  StructField("Age", IntegerType(), True)
])

customer_df = spark.read.format("csv").option("header", "true").schema(customer_schema).load(f"{customer_data_path}")

# Removing Duplicates
customer_df_dup_rem = customer_df.drop_duplicates(["Customer_ID", "Customer_Name"])

# Adding Ingestion Timestamp
customer_df_dup_rem = customer_df_dup_rem.withColumn("Customer_Ingested_At", current_timestamp())

# Saving the Bronze Table
customer_df_dup_rem.write.mode("overwrite").format("delta").saveAsTable("customer_bronze")

In [0]:
# Loading Products raw data
dbutils.widgets.text("products_data_path", "/Volumes/workspace/retail_schema/retail_raw_data/master_product.csv")
products_data_path = dbutils.widgets.get("products_data_path")

products_schema = StructType([
  StructField("Product_ID", StringType(), True),
  StructField("Category", StringType(), True),
  StructField("Sub_Category", StringType(), True),
  StructField("Product_Name", StringType(), True),
])

products_df = spark.read.format("csv").option("header", "true").schema(products_schema).load(f"{products_data_path}")

# Removing Duplicates
products_df_dup_rem = products_df.drop_duplicates(["Product_ID", "Product_Name"])

# Adding Metadata Columns
products_df_dup_rem = products_df_dup_rem.withColumn("Product_Ingested_At", current_timestamp())

# Saving the Bronze Table
products_df_dup_rem.write.mode("overwrite").format("delta").saveAsTable("products_bronze")

In [0]:
# Loading Store raw data
dbutils.widgets.text("stores_data_path", "/Volumes/workspace/retail_schema/retail_raw_data/store_data.csv")
stores_data_path = dbutils.widgets.get("stores_data_path")

store_schema = StructType([
  StructField("Row_ID", IntegerType(), True),
  StructField("Order_ID", StringType(), True),
  StructField("Order_Date", DateType(), True),
  StructField("Ship_Date", DateType(), True),
  StructField("Ship_Mode", StringType(), True),
  StructField("Customer_ID", StringType(), True),
  StructField("Product_ID", StringType(), True),
  StructField("Sales", DoubleType(), True),
  StructField("Discount", DoubleType(), True)
])

store_df = spark.read.format("csv").option("header", "true").option("dateFormat", "M/d/yyyy").schema(store_schema).load(f"{stores_data_path}")

# Removing Duplicates
store_df_dup_rem = store_df.drop_duplicates(["Row_ID"])

# Adding Metadata Columns
store_df_dup_rem = store_df_dup_rem.withColumn("Store_Ingested_At", current_timestamp())

# Saving the Bronze Table
store_df_dup_rem.write.mode("overwrite").format("delta").saveAsTable("store_bronze")